# Business Entity Resolution — Local Pipeline (VS Code / Jupyter)

Runs the full pipeline locally: blocking → features → train → infer → validate.
No AWS/SageMaker needed. Designed to run in VS Code's Jupyter extension —
open this file in VS Code, select a Python kernel (3.9+), and run cells top to bottom.

**Before running:** put these four files somewhere on disk and point `DATA_DIR`
at that folder below:
- `train_source1.tsv`, `train_source2.tsv`, `train_source3.tsv`, `train_ground_truth.tsv`
- `test_source1.tsv`, `test_source2.tsv`, `test_source3.tsv`

**Two blocking modes** (set `USE_EMBEDDINGS` below):
- `False` (default) — network-free token-index blocking. No model download,
  works offline, proven on the real 11.7M-record test set already.
- `True` — embedding-based blocking (sentence-transformers + FAISS). Better
  recall, especially for cross-script name matches (e.g. Devanagari vs.
  transliterated Latin business names), but needs internet access to
  download the model the first time and `pip install sentence-transformers faiss-cpu`.


In [5]:
# ── 0. Install dependencies (run once) ─────────────────────────────────────
%pip install -q pandas numpy scikit-learn lightgbm rapidfuzz pyarrow
# only needed if you set USE_EMBEDDINGS = True below:
# %pip install -q sentence-transformers faiss-cpu


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# ── 1. Config — EDIT THESE PATHS ────────────────────────────────────────────
import os, sys, multiprocessing as mp

DATA_DIR = "C:/Users/muddu/Downloads/ser_pipeline(1)/data"         # folder containing the 7 challenge TSVs
SRC_DIR = "../src"   # the src/ folder from the pipeline package
OUT_DIR = "./output"

USE_EMBEDDINGS = False        # True = blocking.py (needs internet), False = blocking_token_index.py
TOP_K = 20                    # candidates per (S1 entity, source). Lower = faster, may cost recall.
N_JOBS = max(1, mp.cpu_count() - 1)   # cores to use for feature engineering

os.makedirs(OUT_DIR, exist_ok=True)
sys.path.insert(0, SRC_DIR)
print(f"Using {N_JOBS} worker processes for feature engineering.")
print(f"Blocking mode: {'embeddings (blocking.py)' if USE_EMBEDDINGS else 'token-index (blocking_token_index.py)'}")


Using 15 worker processes for feature engineering.
Blocking mode: token-index (blocking_token_index.py)


In [7]:
import os

train_cand_path = f"{OUT_DIR}/train_candidate_pairs.tsv"
test_cand_path = f"{OUT_DIR}/test_candidate_pairs.tsv"
train_feat_path = f"{OUT_DIR}/train_features.parquet"
test_feat_path = f"{OUT_DIR}/test_features.parquet"
model_path = f"{OUT_DIR}/model.txt"
threshold_path = f"{OUT_DIR}/threshold.json"

for name, path in [("train_cand_path", train_cand_path), ("test_cand_path", test_cand_path),
                    ("train_feat_path", train_feat_path), ("test_feat_path", test_feat_path),
                    ("model_path", model_path), ("threshold_path", threshold_path)]:
    print(f"{name}: {'FOUND' if os.path.exists(path) else 'missing'} -> {path}")

print()
print("Run the notebook cells starting from the first 'missing' item above.")

train_cand_path: FOUND -> ./output/train_candidate_pairs.tsv
test_cand_path: FOUND -> ./output/test_candidate_pairs.tsv
train_feat_path: missing -> ./output/train_features.parquet
test_feat_path: missing -> ./output/test_features.parquet
model_path: missing -> ./output/model.txt
threshold_path: missing -> ./output/threshold.json

Run the notebook cells starting from the first 'missing' item above.


## 2. Blocking / candidate generation

Runs once for TRAIN (to build labeled training pairs) and once for TEST
(to build the candidates you'll actually score). Token-index mode processes
one country at a time to keep memory bounded — this is the same approach
already validated end-to-end on the real 11.7M-record test set.

In [8]:
# ── 2a. Token-index blocking (default, offline) ─────────────────────────────
from blocking_token_index import build_candidates_for_country
import pandas as pd
import os

def run_token_blocking(split: str):
    """split = 'train' or 'test'. Loops over the countries present in
    source1 and merges into one candidate_pairs file for that split.
    Skips recomputation if the output file already exists on disk —
    safe to re-run after a kernel restart without redoing the slow part."""
    out_path = f"{OUT_DIR}/{split}_candidate_pairs.tsv"
    if os.path.exists(out_path):
        print(f"[{split}] {out_path} already exists — skipping recompute. "
              f"Delete the file first if you want to force a rebuild.")
        return out_path

    s1_path = f"{DATA_DIR}/{split}_source1.tsv"
    s2_path = f"{DATA_DIR}/{split}_source2.tsv"
    s3_path = f"{DATA_DIR}/{split}_source3.tsv"

    s1_full = pd.read_csv(s1_path, sep="\t", dtype=str)
    countries = s1_full["country"].dropna().str.strip().unique().tolist()
    print(f"[{split}] countries found: {countries}")

    lookup = {}
    for country in countries:
        result = build_candidates_for_country(s1_path, s2_path, s3_path, country, top_k=TOP_K)
        for s1_id, ids_str in zip(result["source1_entity_id"], result["candidate_entity_ids"]):
            lookup[s1_id] = ids_str

    rows = [{"source1_entity_id": eid, "candidate_entity_ids": lookup.get(eid, "")}
            for eid in s1_full["entity_id"]]
    pd.DataFrame(rows).to_csv(out_path, sep="\t", index=False)
    print(f"[{split}] wrote {out_path}")
    return out_path

if not USE_EMBEDDINGS:
    train_cand_path = run_token_blocking("train")
    test_cand_path = run_token_blocking("test")

[train] ./output/train_candidate_pairs.tsv already exists — skipping recompute. Delete the file first if you want to force a rebuild.
[test] ./output/test_candidate_pairs.tsv already exists — skipping recompute. Delete the file first if you want to force a rebuild.


In [9]:
# ── 2b. Embedding-based blocking (optional, needs internet) ─────────────────
if USE_EMBEDDINGS:
    from blocking import build_candidates

    train_cand_path = f"{OUT_DIR}/train_candidate_pairs.tsv"
    test_cand_path = f"{OUT_DIR}/test_candidate_pairs.tsv"
    build_candidates(f"{DATA_DIR}/train_source1.tsv", f"{DATA_DIR}/train_source2.tsv",
                      f"{DATA_DIR}/train_source3.tsv", train_cand_path, top_k=TOP_K)
    build_candidates(f"{DATA_DIR}/test_source1.tsv", f"{DATA_DIR}/test_source2.tsv",
                      f"{DATA_DIR}/test_source3.tsv", test_cand_path, top_k=TOP_K)

## 3. Feature engineering

Uses all CPU cores (`N_JOBS`) — this was the main local-performance
bottleneck in the original version (row-wise `DataFrame.apply` doesn't
scale to tens of millions of pairs), so it's since been rewritten to use
plain dict lookups + multiprocessing.

In [ ]:
from features import _load, explode_candidates, build_features
import os

def build_features_for_split(split: str, cand_path: str):
    out_path = f"{OUT_DIR}/{split}_features.parquet"
    if os.path.exists(out_path):
        print(f"[{split}] {out_path} already exists — skipping recompute. "
              f"Delete the file first if you want to force a rebuild.")
        return out_path

    s1 = _load(f"{DATA_DIR}/{split}_source1.tsv")
    s2 = _load(f"{DATA_DIR}/{split}_source2.tsv")
    s3 = _load(f"{DATA_DIR}/{split}_source3.tsv")
    other = {**s2, **s3}

    pairs = explode_candidates(cand_path)
    print(f"[{split}] {len(pairs)} candidate pairs to featurize")
    feats = build_features(pairs, s1, other, n_jobs=N_JOBS)

    feats.to_parquet(out_path, index=False)
    print(f"[{split}] wrote {out_path}")
    return out_path

train_feat_path = build_features_for_split("train", train_cand_path)

[train] 56518133 candidate pairs to featurize


In [ ]:
test_feat_path = build_features_for_split("test", test_cand_path)


## 4. Train the matching model

LightGBM classifier on labeled candidate pairs, with the decision
threshold swept directly against the competition's own macro **F_0.5**
formula (not accuracy/AUC), since that's precision-heavy and what's
actually scored.

In [ ]:
from train import main as train_main

model_path = f"{OUT_DIR}/model.txt"
threshold_path = f"{OUT_DIR}/threshold.json"

train_main(
    features_path=train_feat_path,
    ground_truth_path=f"{DATA_DIR}/train_ground_truth.tsv",
    model_out=model_path,
    threshold_out=threshold_path,
)


## 5. Inference — produces `matching_results.tsv`

This is the file that gets uploaded to the leaderboard.

In [ ]:
from infer import main as infer_main

infer_main(
    test_source1_path=f"{DATA_DIR}/test_source1.tsv",
    test_features_path=test_feat_path,
    test_candidates_path=test_cand_path,
    model_path=model_path,
    threshold_path=threshold_path,
    out_dir=OUT_DIR,
)
print("Done. See:", f"{OUT_DIR}/matching_results.tsv", "and", f"{OUT_DIR}/candidate_pairs.tsv")


## 6. Sanity-check the output format

Mirrors the challenge's own `utils/validate_submission.py` rules — run
that script too before actually submitting, this is just a fast local check.

In [ ]:
import pandas as pd

matches = pd.read_csv(f"{OUT_DIR}/matching_results.tsv", sep="\t", dtype=str).fillna("")
s1_test = pd.read_csv(f"{DATA_DIR}/test_source1.tsv", sep="\t", dtype=str)
valid_ids = set(pd.read_csv(f"{DATA_DIR}/test_source2.tsv", sep="\t", dtype=str, usecols=["entity_id"])["entity_id"]) \
          | set(pd.read_csv(f"{DATA_DIR}/test_source3.tsv", sep="\t", dtype=str, usecols=["entity_id"])["entity_id"])

assert len(matches) == len(s1_test), "row count mismatch vs test_source1"
assert matches["source1_entity_id"].is_unique, "duplicate source1_entity_id rows"
assert set(matches["source1_entity_id"]) == set(s1_test["entity_id"]), "missing/extra S1 entities"

bad_rows, dupe_rows, n_matched = 0, 0, 0
for ids_str in matches["matched_entity_ids"]:
    if not ids_str:
        continue
    ids = ids_str.split(",")
    n_matched += len(ids)
    if len(ids) != len(set(ids)):
        dupe_rows += 1
    if any(i not in valid_ids for i in ids):
        bad_rows += 1

print(f"Rows: {len(matches)} (matches test_source1 exactly)")
print(f"Rows with invalid candidate IDs: {bad_rows} (should be 0)")
print(f"Rows with internal duplicate IDs: {dupe_rows} (should be 0)")
print(f"Total predicted matches: {n_matched}")
print(f"Entities predicted as singletons (no match): {(matches['matched_entity_ids']=='').sum()}")
